# 01 Cleaning: Toronto Building Permits (2017 onward)

**Goal of this notebook:** turn the two raw City of Toronto files (Active Permits and Cleared Permits) into one clean table, `data/clean/permits_clean.csv`, with one row per permit, ready for analysis.

**Steps**
1. Load both raw files and check their columns match
2. Combine them and record which file each row came from
3. Parse dates and keep applications from 2017 onward
4. Remove duplicates and keep the latest revision per permit number
5. Clean declared construction cost
6. Derive FSA, application year and month, and approval days
7. Group permit and work types into clean categories
8. Save the clean file and print the cleaning log

Every step that drops or changes rows is recorded with `log_step()`, and the final cell prints a table to paste into `docs/cleaning_log.md`.

> Cells marked **CHECK** print something you should look at before trusting the next step. Where I could not confirm the data's behaviour in advance, the cell says so.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Paths are relative to the repo root. The notebook lives in notebooks/,
# so the root is one folder up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

SCOPE_START = pd.Timestamp("2017-01-01")

# Cleaning log: one entry per step, so every row we drop is accounted for.
cleaning_log = []

def log_step(step, rows_before, rows_after, note=""):
    # If a cell is re-run, replace its earlier entry instead of adding a duplicate.
    cleaning_log[:] = [e for e in cleaning_log if e["step"] != step]
    cleaning_log.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "note": note,
    })
    print(f"{step}: {rows_before:,} -> {rows_after:,} ({rows_before - rows_after:,} removed)")

## 1. Load the raw files

Everything is read as text (`dtype=str`) on purpose. Permit numbers and postal codes are identifiers, not numbers, and the cost column contains placeholder text in some rows. Reading as text means nothing is silently converted; we convert each column ourselves, on purpose, later on.

In [2]:
active = pd.read_csv(RAW / "active_permits.csv", dtype=str, low_memory=False)
cleared = pd.read_csv(RAW / "cleared_permits_since_2017.csv", dtype=str, low_memory=False)

print("Active: ", active.shape)
print("Cleared:", cleared.shape)

Active:  (202298, 32)
Cleared: (443717, 32)


In [3]:
# CHECK: the two files should have the same columns. Anything printed here
# needs a decision before combining.
only_active = set(active.columns) - set(cleared.columns)
only_cleared = set(cleared.columns) - set(active.columns)
print("Only in Active: ", sorted(only_active))
print("Only in Cleared:", sorted(only_cleared))
print()
print(list(active.columns))

Only in Active:  []
Only in Cleared: []

['_id', 'PERMIT_NUM', 'REVISION_NUM', 'PERMIT_TYPE', 'STRUCTURE_TYPE', 'WORK', 'STREET_NUM', 'STREET_NAME', 'STREET_TYPE', 'STREET_DIRECTION', 'POSTAL', 'GEO_ID', 'WARD_GRID', 'APPLICATION_DATE', 'ISSUED_DATE', 'COMPLETED_DATE', 'STATUS', 'DESCRIPTION', 'CURRENT_USE', 'PROPOSED_USE', 'DWELLING_UNITS_CREATED', 'DWELLING_UNITS_LOST', 'EST_CONST_COST', 'ASSEMBLY', 'INSTITUTIONAL', 'RESIDENTIAL', 'BUSINESS_AND_PERSONAL_SERVICES', 'MERCANTILE', 'INDUSTRIAL', 'INTERIOR_ALTERATIONS', 'DEMOLITION', 'BUILDER_NAME']


## 2. Combine

We tag each row with its source file before stacking, because the same permit can appear in both files (for example, a permit that was active when one snapshot was taken and cleared later). The tag lets us decide which copy to keep in step 4.

`_id` is a row number added by the portal's datastore. It means nothing across files, so we drop it.

In [4]:
active["source"] = "active"
cleared["source"] = "cleared"

permits = pd.concat([active, cleared], ignore_index=True)
permits = permits.drop(columns=["_id"], errors="ignore")

# Trim stray spaces in every text column. Values like "M5V " and "M5V"
# would otherwise count as two different FSAs.
text_cols = permits.select_dtypes(include="object").columns
permits[text_cols] = permits[text_cols].apply(lambda s: s.str.strip())

rows_start = len(permits)
log_step("Combine Active + Cleared", len(active) + len(cleared), rows_start)

C:\Users\Mahmo\AppData\Local\Temp\ipykernel_37568\407648854.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = permits.select_dtypes(include="object").columns


Combine Active + Cleared: 646,015 -> 646,015 (0 removed)


## 3. Dates and the 2017+ scope

The Cleared file is named "since 2017", but a quick look at the portal's API showed it also contains permits *applied for* long before 2017 (for example 1986 and 2005 applications). So "since 2017" seems to refer to when permits were cleared, not when they were applied for. Our scope is by **application date**, so we filter on that.

In [5]:
for col in ["APPLICATION_DATE", "ISSUED_DATE", "COMPLETED_DATE"]:
    # errors="coerce" turns unparseable values into NaT so we can count them
    # instead of the notebook crashing.
    permits[col] = pd.to_datetime(permits[col], errors="coerce")

# CHECK: how many dates are missing or unparseable?
print(permits[["APPLICATION_DATE", "ISSUED_DATE", "COMPLETED_DATE"]].isna().sum())
print()
print("Application date range:", permits["APPLICATION_DATE"].min(), "to", permits["APPLICATION_DATE"].max())

APPLICATION_DATE         6
ISSUED_DATE          49900
COMPLETED_DATE      202298
dtype: int64

Application date range: 1979-06-21 00:00:00 to 2026-10-09 00:00:00


In [6]:
before = len(permits)
permits = permits[permits["APPLICATION_DATE"] >= SCOPE_START].copy()
log_step("Keep applications from 2017-01-01 onward", before, len(permits),
         "Rows with a missing application date are also removed here")

Keep applications from 2017-01-01 onward: 646,015 -> 437,705 (208,310 removed)


## 4. Duplicates and latest revision

Three kinds of repetition are possible:
- **Exact duplicate rows** (identical in every column). Safe to drop.
- **Same permit and revision in both files.** We keep the **Cleared** copy. The first full run found 761 such pairs. In every sampled pair the Cleared copy had a final status (Closed or Cancelled) and a completed date, while the Active copy still said "Inspection". The Active copy's `ISSUED_DATE` also matched the Cleared copy's `COMPLETED_DATE`, so the Active copy looks stale and its issued date unreliable. Keeping Cleared is the safer choice.
- **Several revisions of one permit.** `REVISION_NUM` is stored as text like "00", "01". We convert it to a number and keep the highest per `PERMIT_NUM`, which is the brief's rule of one row per permit at its latest revision.

In [7]:
before = len(permits)
permits = permits.drop_duplicates(subset=[c for c in permits.columns if c != "source"])
log_step("Drop exact duplicate rows", before, len(permits))

Drop exact duplicate rows: 437,705 -> 437,705 (0 removed)


In [8]:
# CHECK: permits that appear in both files at the same revision.
key = ["PERMIT_NUM", "REVISION_NUM"]
in_both = permits.groupby(key)["source"].nunique()
in_both = in_both[in_both > 1].index
print(f"Permit + revision pairs in both files: {len(in_both):,}")

sample = permits.set_index(key).loc[in_both[:5]].reset_index()
sample[key + ["source", "STATUS", "ISSUED_DATE", "COMPLETED_DATE"]]

Permit + revision pairs in both files: 761


,PERMIT_NUM,REVISION_NUM,source,STATUS,ISSUED_DATE,COMPLETED_DATE
0,14 263052 STS,00,active,Inspection,2019-04-16,NaT
1,14 263052 STS,00,cleared,Closed,2018-05-24,2019-04-16
2,17 101592 BLD,00,active,Inspection,2020-01-10,NaT
3,17 101592 BLD,00,cleared,Cancelled,2019-03-01,2020-01-10
4,17 123207 BLD,00,active,Inspection,2021-02-01,NaT
5,17 123207 BLD,00,cleared,Closed,2017-04-03,2021-02-01
6,17 129930 BLD,00,active,Inspection,2017-11-16,NaT
7,17 129930 BLD,00,cleared,Closed,2017-07-12,2017-11-16
8,17 132098 BLD,00,active,Inspection,2020-10-27,NaT
9,17 132098 BLD,00,cleared,Closed,2019-05-30,2020-07-15


In [9]:
permits["revision_int"] = pd.to_numeric(permits["REVISION_NUM"], errors="coerce")
print("Revision numbers that are not numeric:", permits["revision_int"].isna().sum())

# Sort so the row we want to keep comes LAST within each permit:
# highest revision, and on a tie, Cleared after Active.
permits["source_rank"] = permits["source"].map({"active": 0, "cleared": 1})
permits = permits.sort_values(["PERMIT_NUM", "revision_int", "source_rank"])

before = len(permits)
permits = permits.drop_duplicates(subset="PERMIT_NUM", keep="last")
log_step("Keep latest revision per PERMIT_NUM (Cleared wins ties)", before, len(permits))

permits = permits.drop(columns=["source_rank"])

Revision numbers that are not numeric: 7
Keep latest revision per PERMIT_NUM (Cleared wins ties): 437,705 -> 388,782 (48,923 removed)


## 5. Declared construction cost

`EST_CONST_COST` is published as text. Some rows hold a placeholder sentence instead of a number (seen in the API: "DO NOT UPDATE OR DELETE THIS INFO FIELD"). These appear to be trade permits (plumbing, HVAC, drains) attached to a main building permit, where the cost is recorded on the main permit. We convert to a number, turn non-numbers into missing, and **do not fill them in**: making up costs would distort the value analysis.

We keep these rows. They still count as permits; they are just excluded from value totals.

**Zero costs.** The first run found about 41,000 permits with a declared cost of exactly 0. A real project rarely costs nothing, so 0 most likely means "not declared". We keep `est_cost` as published, and add `cost_declared` (True when cost is above 0). Value analyses should use only rows where `cost_declared` is True.

**Very large costs.** The maximum was 1,000,000,000. That could be a real major project or a typing error, so the CHECK cell lists the largest values with their descriptions to judge by eye.

In [10]:
raw_cost = permits["EST_CONST_COST"]
permits["est_cost"] = pd.to_numeric(raw_cost.str.replace(",", "", regex=False), errors="coerce")

# CHECK: what non-numeric values were there?
non_numeric = raw_cost[permits["est_cost"].isna() & raw_cost.notna()]
print("Non-numeric cost values (top 10):")
print(non_numeric.value_counts().head(10))
print()
print("Missing cost after conversion:", permits["est_cost"].isna().sum())
print("Cost exactly 0:              ", (permits["est_cost"] == 0).sum())
print("Negative cost:               ", (permits["est_cost"] < 0).sum())

# 0 is treated as "not declared" for value analysis (see note above).
permits["cost_declared"] = permits["est_cost"] > 0
print("Rows with a declared cost (> 0):", permits["cost_declared"].sum())
permits["est_cost"].describe()

Non-numeric cost values (top 10):
EST_CONST_COST
DO NOT UPDATE OR DELETE THIS INFO FIELD    209700
Name: count, dtype: int64

Missing cost after conversion: 219380
Cost exactly 0:               40854
Negative cost:                0
Rows with a declared cost (> 0): 128548


count    1.694020e+05
mean     6.406227e+05
std      9.963448e+06
min      0.000000e+00
25%      1.000000e+03
50%      2.500000e+04
75%      1.500000e+05
max      1.000000e+09
Name: est_cost, dtype: float64

In [11]:
# CHECK: the 15 largest declared costs. Do the descriptions fit the amounts?
# A house renovation at $1 billion would be a typo; a tower may be real.
cols = ["PERMIT_NUM", "PERMIT_TYPE", "STRUCTURE_TYPE", "WORK", "est_cost", "DESCRIPTION"]
with pd.option_context("display.max_colwidth", 120):
    display(permits.nlargest(15, "est_cost")[cols])

,PERMIT_NUM,PERMIT_TYPE,STRUCTURE_TYPE,WORK,est_cost,DESCRIPTION
181081,23 120705 BLD,New Building,Mixed Use/Res w Non Res,New Building,1.000000e+09,Proposal for the construction of a 39-storey purpose-built rental tower 'B' including a 10-storey mixed-used podium ...
280884,24 225558 FSU,Fire/Security Upgrade,Office,Sprinklers,1.000000e+09,"Sprinkler permit update based on 24-181473 ALT, new permit since previous 20-113679 permit is closed.Macdonald Block..."
581575,25 111279 BLD,New Building,Mixed Use/Res w Non Res,New Building,1.000000e+09,"The proposed development consists of a 38 storey mixed use development, containing 498 units."
34872,25 124438 BLD,New Building,Industrial,New Building,8.050000e+08,"In support of the Fire Ventilation Upgrade Program, TTC will be carrying out installation of fan plants at North and..."
97411,23 199437 BLD,New Building,"Transit Station,Subway, Bus Terminal",New Building,7.000000e+08,Construct new station for Ontario Line - Exhibition Station
63882,24 178413 BLD,New Building,"Transit Station,Subway, Bus Terminal",New Building,6.300756e+08,Construction of the Sheppard East transit station for the SSE-SRS project.
153687,25 257691 BLD,New Building,Mixed Use/Res w Non Res,New Building,6.010000e+08,TOWER A (with podium and underground garage): Proposed 39 Storey building with 357 rental residential units and ret...
85699,25 259382 BLD,New Building,Mixed Use/Res w Non Res,New Building,6.010000e+08,TOWER B - Proposed 46 Storey rental residential building with 461 residential units and retail units
181184,24 136495 BLD,New Building,"Transit Station,Subway, Bus Terminal",New Building,5.551059e+08,Construction of the Lawrence East Station for the SSE-SRS project
581935,25 126623 BLD,New Building,Apartment Building,New Building,5.150000e+08,"Construction of a new 62 storey multi residential condominium building with 749 dwelling units, three levels of und..."


## 6. Derived columns: FSA, year, month, approval days

**FSA.** In the API sample, `POSTAL` already held only three characters (like "M5V"). Taking the first three characters works whether the file has the FSA or a full postal code. A valid Toronto FSA is the letter M, a digit, then a letter, so anything else is flagged as invalid and set to missing (the permit is kept, but it cannot be mapped).

**Approval days** = issued date minus application date. It only exists for permits that have been issued. Negative values would be data errors and are set to missing.

**Open question to check:** after step 4 each permit is at its latest revision. I do not yet know whether a revision row carries the *original* application date or the date the revision was applied for. If it is the revision date, approval days for revised permits could be wrong (even negative). The negative count printed below is the first clue; if it is large, we should take the application date from revision "00" instead.

In [12]:
permits["fsa"] = permits["POSTAL"].str.upper().str[:3]
valid_fsa = permits["fsa"].str.fullmatch(r"M\d[A-Z]", na=False)

# CHECK: what do the invalid values look like?
print("Invalid or missing FSA:", (~valid_fsa).sum())
print(permits.loc[~valid_fsa, "POSTAL"].value_counts(dropna=False).head(10))
permits.loc[~valid_fsa, "fsa"] = np.nan

# The first run found about 24,600 blank postal codes. These permits are kept
# (they still count city-wide) but cannot appear on the FSA map. If blanks are
# concentrated in one permit type, the map would under-count that type, which
# is worth stating as a limitation.
print()
print("Share of each PERMIT_TYPE with no FSA (top 10):")
print(permits.groupby("PERMIT_TYPE")["fsa"].apply(lambda s: s.isna().mean())
      .sort_values(ascending=False).head(10).round(3))

permits["app_year"] = permits["APPLICATION_DATE"].dt.year
permits["app_month"] = permits["APPLICATION_DATE"].dt.month
# First day of the month: handy for monthly trend charts in Tableau.
permits["app_year_month"] = permits["APPLICATION_DATE"].dt.to_period("M").dt.to_timestamp()

permits["approval_days"] = (permits["ISSUED_DATE"] - permits["APPLICATION_DATE"]).dt.days
neg = permits["approval_days"] < 0
print("Negative approval days (set to missing):", neg.sum())
permits.loc[neg, "approval_days"] = np.nan

permits["approval_days"].describe()

Invalid or missing FSA: 24630
POSTAL
    24630
Name: count, dtype: int64

Share of each PERMIT_TYPE with no FSA (top 10):
PERMIT_TYPE
Toronto Buildings Contacts               1.000
Toronto Building Standard Attachments    1.000
Conditional Permit                       0.410
New Building                             0.218
Temporary Structures                     0.179
AS Alternative Solution                  0.174
Partial Permit                           0.142
Designated Structures                    0.139
Fire/Security Upgrade                    0.108
Building Additions/Alterations           0.106
Name: fsa, dtype: float64
Negative approval days (set to missing): 2


count    345601.000000
mean         70.051334
std         156.925940
min           0.000000
25%          10.000000
50%          28.000000
75%          58.000000
max        3409.000000
Name: approval_days, dtype: float64

In [13]:
# CHECK: do revised permits (revision 01 and up) look different?
# If a revision row carries the date the REVISION was applied for, its
# approval days would measure the revision, not the original permit, and
# would look much shorter than for revision 00.
permits["is_revision"] = permits["revision_int"] > 0
print(permits.groupby("is_revision")["approval_days"].describe().round(1))

                count  mean    std  min   25%   50%   75%     max
is_revision                                                      
False        306572.0  72.6  160.2  0.0  10.0  29.0  61.0  3409.0
True          39029.0  49.8  126.4  0.0  11.0  23.0  41.0  3061.0


In [14]:
# CHECK: status values. Earlier output showed "Cancelled" and "Superseded".
# A cancelled permit may never lead to material purchases, so we flag it
# rather than drop it; the analysis can then show results with and without.
print(permits["STATUS"].value_counts(dropna=False).to_string())
permits["is_cancelled"] = permits["STATUS"].str.contains("cancel", case=False, na=False)
print()
print("Flagged as cancelled:", permits["is_cancelled"].sum())

STATUS
Closed                          244529
Inspection                       52582
Cancelled                        32999
Permit Issued                    28635
Revision Issued                  10991
Revocation Pending                3255
Examiner's Notice Sent            2757
Issuance Pending                  2723
Under Review                      1890
Approved                          1763
Not Started                       1065
Application On Hold               1047
Refusal Notice                     850
Application Acceptable             743
Superseded                         571
Work Not Started                   537
Open                               482
Pending Cancellation               325
Response Received                  273
Ready for Issuance                 270
Deficiency Notice Issued           117
Application Received                97
Abandoned                           88
Not Started - Express               63
Refused                             52
Pending Parent Fol

## 7. Categories

The first full run printed every `PERMIT_TYPE` (21 values) and the most common `STRUCTURE_TYPE` (130 values) and `WORK` (106 values). The mappings below are built from those real values instead of guesses.

Three findings shaped them:
1. **A few permit types are admin records, not construction** (deferred development charge fees, contact records, attachments, converted historical data). There are only a few hundred rows; they are removed and logged.
2. **About 175,000 rows are trade permits tied to a building permit**: plumbing, HVAC, drains and fire services, with `WORK` like "Building Permit Related(PS)". The permit number suggests how they link: "24 110570 HVA" and "24 110570 DRN" share the middle number. *This link is inferred from the number format, not from City documentation.* We keep these rows, flag them as `is_related_permit`, and add `project_id` (year + number, without the type code), so the analysis can count projects instead of inflating counts with related permits.
3. **`WORK` is specific but `PERMIT_TYPE` is a better fallback.** We map known `WORK` values first, then fall back to `PERMIT_TYPE`, then "Other".

In [15]:
# Admin record types: not construction activity.
ADMIN_PERMIT_TYPES = [
    "DCs DeferredFees",
    "Toronto Buildings Contacts",
    "Toronto Building Standard Attachments",
    "Building Historical data - Converted",
]
before = len(permits)
permits = permits[~permits["PERMIT_TYPE"].isin(ADMIN_PERMIT_TYPES)].copy()
log_step("Drop admin record types (not construction)", before, len(permits),
         ", ".join(ADMIN_PERMIT_TYPES))

Drop admin record types (not construction): 388,782 -> 388,262 (520 removed)


In [16]:
# project_id: the permit number without its last part (the type code).
# "24 110570 HVA" -> "24 110570". Related permits for one project share it.
permits["project_id"] = permits["PERMIT_NUM"].str.rsplit(" ", n=1).str[0]

# Trade permits filed alongside a building permit.
permits["is_related_permit"] = permits["WORK"].str.startswith("Building Permit Related", na=False)

print("Related (trade) permits:", permits["is_related_permit"].sum())
print("Distinct permits:  ", permits["PERMIT_NUM"].nunique())
print("Distinct projects: ", permits["project_id"].nunique())

Related (trade) permits: 175371
Distinct permits:   388262
Distinct projects:  206516


In [17]:
# ---- work_category ----
# Step 1: specific WORK values. Only values seen in the first full run.
WORK_MAP = {
    # New construction
    "New Building": "New construction",
    "New Laneway / Rear Yard Suite": "New construction",
    "Partial Permit - Foundation": "New construction",
    "Partial Permit - Structural Framing": "New construction",
    # Demolition
    "Demolition": "Demolition",
    # Additions and new structures on an existing property
    "Addition(s)": "Addition",
    "Garage": "Addition",
    "Accessory Building(s)": "Addition",
    "Deck": "Addition",
    "Porch": "Addition",
    # Work inside or on an existing building
    "Interior Alterations": "Renovation / alteration",
    "Underpinning": "Renovation / alteration",
    "Second Suite (New)": "Renovation / alteration",  # new unit inside an existing house
    "Walk-Out Stair": "Renovation / alteration",
    "Garage Repair/Reconstruction": "Renovation / alteration",
    "Balcony/Guard Repairs": "Renovation / alteration",
    "Fire Damage": "Renovation / alteration",
    # Building services (plumbing, HVAC, drains, fire and security systems)
    "Building Permit Related(PS)": "Building services",
    "Building Permit Related(MS)": "Building services",
    "Building Permit Related (DR)": "Building services",
    "Building Permit Related(FS)": "Building services",
    "Back Water Valve (Sewer only)": "Building services",
    "Backflow Prevention Devices (Water only)": "Building services",
    "Inside and Outside Drains": "Building services",
    "Site Service": "Building services",
    "Piping: Other Buildings": "Building services",
    "Sprinklers": "Building services",
    "Fire Alarm": "Building services",
    "Electromagnetic Locks": "Building services",
    # Not a construction work type for our purpose
    "Sign Building Permit Related": "Other",
    "Party Wall Admin Permits": "Other",
    "Alternative Solution": "Other",
    "Temporary Structures": "Other",
    "Certified Tents": "Other",
    "Pool Fence Enclosure": "Other",
}

# Step 2: fallback by PERMIT_TYPE for WORK values not listed above
# (for example "Multiple Projects" or "Other(BA)").
PERMIT_TYPE_FALLBACK = {
    "New Houses": "New construction",
    "New Building": "New construction",
    "Partial Permit": "New construction",   # staged permits for large new buildings (assumption)
    "Demolition Folder (DM)": "Demolition",
    "Building Additions/Alterations": "Renovation / alteration",
    "Small Residential Projects": "Renovation / alteration",
    "Plumbing(PS)": "Building services",
    "Mechanical(MS)": "Building services",
    "Drain and Site Service": "Building services",
    "Fire/Security Upgrade": "Building services",
}

permits["work_category"] = (
    permits["WORK"].map(WORK_MAP)
    .fillna(permits["PERMIT_TYPE"].map(PERMIT_TYPE_FALLBACK))
    .fillna("Other")
)
print(permits["work_category"].value_counts())

work_category
Building services          219369
Renovation / alteration     94900
Other                       24020
New construction            22705
Addition                    15783
Demolition                  11485
Name: count, dtype: int64


In [18]:
# CHECK: how each PERMIT_TYPE ends up split across categories.
# Look for anything surprising, e.g. New Houses landing in Building services.
pd.crosstab(permits["PERMIT_TYPE"], permits["work_category"])

work_category,Addition,Building services,Demolition,New construction,Other,Renovation / alteration
PERMIT_TYPE,,,,,,
AS Alternative Solution,0,0,0,0,2334,0
Building Additions/Alterations,859,0,0,0,0,44572
Conditional Permit,3,116,0,888,146,38
Demolition Folder (DM),0,0,11485,0,0,0
Designated Structures,0,0,0,0,11127,0
Drain and Site Service,0,53286,0,0,0,0
Fire/Security Upgrade,0,8978,0,0,0,0
Mechanical(MS),0,70398,0,0,0,0
New Building,0,0,0,1993,0,0


In [19]:
# ---- sector ----
# STRUCTURE_TYPE has 130 values, so we use ordered keyword rules.
# Order matters: "Multiple Use/Non Residential" contains the word
# "residential", so the non-residential check must come first.
def map_sector(structure_type):
    if pd.isna(structure_type):
        return "Unknown"
    s = structure_type.lower()
    if s in ("other", "first party"):          # meaning unclear, do not guess
        return "Unknown"
    if "non residential" in s or "non-residential" in s:
        return "Non-residential"
    if "mixed use" in s:
        return "Mixed use"
    residential_words = ["sfd", "unit -", "townhouse", "apartment", "multiple unit",
                         "laneway", "rear yard suite", "residential"]
    if any(w in s for w in residential_words):
        return "Residential"
    return "Non-residential"

sector_lookup = {v: map_sector(v) for v in permits["STRUCTURE_TYPE"].dropna().unique()}
permits["sector"] = permits["STRUCTURE_TYPE"].map(sector_lookup).fillna("Unknown")
print(permits["sector"].value_counts())

sector
Residential        270665
Non-residential     96220
Unknown             18292
Mixed use            3085
Name: count, dtype: int64


In [20]:
# CHECK: every STRUCTURE_TYPE value and the sector it was given.
# Scan the full list once; any wrong ones go in SECTOR_OVERRIDES and re-run.
SECTOR_OVERRIDES = {}  # e.g. {"Some Value": "Residential"}
permits["sector"] = permits["STRUCTURE_TYPE"].map({**sector_lookup, **SECTOR_OVERRIDES}).fillna("Unknown")

with pd.option_context("display.max_rows", 200):
    display(permits.groupby(["sector", "STRUCTURE_TYPE"]).size()
            .rename("rows").reset_index().sort_values(["sector", "rows"], ascending=[True, False]))

,sector,STRUCTURE_TYPE,rows
0,Mixed use,Mixed Use/Res w Non Res,3085
65,Non-residential,Office,30985
87,Non-residential,Retail Store,12071
84,Non-residential,Restaurant 30 Seats or Less,7272
61,Non-residential,Multiple Use/Non Residential,7080
85,Non-residential,Restaurant Greater Than 30 Seats,4600
42,Non-residential,Industrial,4400
58,Non-residential,Medical/Dental Office,3287
39,Non-residential,Hospital,2455
26,Non-residential,Elementary School,2424


## 8. Save and print the cleaning log

In [21]:
keep_cols = [
    "PERMIT_NUM", "REVISION_NUM", "PERMIT_TYPE", "STRUCTURE_TYPE", "WORK",
    "STATUS", "CURRENT_USE", "PROPOSED_USE", "DESCRIPTION",
    "POSTAL", "fsa", "WARD_GRID", "GEO_ID",
    "APPLICATION_DATE", "ISSUED_DATE", "COMPLETED_DATE",
    "app_year", "app_month", "app_year_month", "approval_days",
    "est_cost", "cost_declared", "is_cancelled", "is_revision", "DWELLING_UNITS_CREATED", "DWELLING_UNITS_LOST",
    "sector", "work_category", "project_id", "is_related_permit", "source",
]
missing = [c for c in keep_cols if c not in permits.columns]
print("Columns not found (check names):", missing)

clean = permits[[c for c in keep_cols if c in permits.columns]]
out = CLEAN / "permits_clean.csv"
clean.to_csv(out, index=False)
print(f"Saved {len(clean):,} rows and {clean.shape[1]} columns to {out}")

Columns not found (check names): []
Saved 388,262 rows and 31 columns to C:\Users\Mahmo\Documents\Projects\toronto-building-permits\data\clean\permits_clean.csv


In [22]:
# Paste this table into docs/cleaning_log.md
log_df = pd.DataFrame(cleaning_log)
try:
    print(log_df.to_markdown(index=False))  # needs the optional "tabulate" package
except ImportError:
    print(log_df.to_string(index=False))

                                                   step  rows_before  rows_after  rows_removed                                                                                                                      note
                               Combine Active + Cleared       646015      646015             0                                                                                                                          
               Keep applications from 2017-01-01 onward       646015      437705        208310                                                                Rows with a missing application date are also removed here
                              Drop exact duplicate rows       437705      437705             0                                                                                                                          
Keep latest revision per PERMIT_NUM (Cleared wins ties)       437705      388782         48923                                      